In [2]:
from huggingface_hub import login
login()

In [3]:
%load_ext autoreload
%autoreload 2

In [ ]:

import src.feat_enc as enc
import torch
from pathlib import Path
import src.stain_norm as sn
import src.class_sampling as cs
import tqdm as notebook_tqdm

NCT = Path("data/NCT-CRC-HE-100K-NONORM")
CRC = Path("data/CRC-VAL-HE-7K")

# 1250/class = 1000 train + 250 val. 1000 is the top of the label-efficiency

train_pool = cs.sample_cohort(NCT, n_per_class=1250, seed=42, strict=True)
mf_nct, sha_nct = cs.write_manifest(
    train_pool, "manifests/nct_1250", NCT,
    {"seed": 42, "n_per_class": 1250, "exclude_classes": "()", "pattern": "*.tif"})

In [4]:
validation_pool = cs.sample_cohort(CRC, n_per_class=250, seed=42, strict=True)
mf_crc, sha_crc = cs.write_manifest(
    validation_pool, "manifests/crc_250", CRC,
    {"seed": 42, "n_per_class": 250, "exclude_classes": "()", "pattern": "*.tif"})

In [11]:
model_h, tf_h, info_h = enc.load_encoder("h_optimus_0")
print(info_h["embed_dim"], info_h["n_params_M"])  
NCT_NOnorm=Path("data/NCT-CRC-HE-100K-NONORM")
paths = sn.stratified_sample(NCT_NOnorm, n_per_class=500, seed=7, exclude_classes=())
labels = [p.parent.name for p in paths]
normalizer = sn.load_normalizer_target("notebooks/output/macenko_target_nct100k.npz")
feats, ok = enc.extract_features(model_h, tf_h, paths, sn.load_image_array,
                                 normalizer=normalizer, batch_size=32)
enc.save_features("features/h_optimus_o_macenko", feats, labels, ok, info_h)

  name         h_optimus_0
  source       hf-hub:bioptimus/H-optimus-0
  note         ViT-G/14, DINOv2-style SSL on H&E WSIs. Native 224. Gated repo.
  embed_dim    1536
  n_params_M   1134.774272
  mean         (0.707223, 0.578729, 0.703617)
  std          (0.211883, 0.230117, 0.177517)
  input_size   (3, 224, 224)
  native_size  (3, 224, 224)
  grid_size    (16, 16)
  n_patches    256
  mean_source  hardcoded (VERIFY)
1536 1134.774272
  32/4500
Normalization failed for data\NCT-CRC-HE-100K-NONORM\BACK\BACK-DIRSWRMP.tif: kthvalue(): Expected reduction dim 0 to have non-zero size.
Normalization failed for data\NCT-CRC-HE-100K-NONORM\BACK\BACK-YCKHWTWL.tif: linalg.eigh: The algorithm failed to converge because the input matrix is ill-conditioned or has too many repeated eigenvalues (error code: 2).
Normalization failed for data\NCT-CRC-HE-100K-NONORM\BACK\BACK-YRFTCQHT.tif: linalg.eigh: The algorithm failed to converge because the input matrix is ill-conditioned or has too many repeated

WindowsPath('features/h_optimus_o_macenko.npz')